## Vraag 1: Correlatie tussen de drie fondsen

**Vraag:** Bewegen IWDA, EMIM en IUSN (small cap) onafhankelijk genoeg om mijn 
small-cap- en EM-allocatie te rechtvaardigen, of bewegen ze sterk gecorreleerd 
mee met de wereldindex?

(IWDA volgt de wereldindex, het is niet een wereldindex zelf. Deze foute gedachtegang had ik zelf eerst ook :p vandaar deze vraagstelling.)

**Aanpak:** Van de slotkoersen (data/prices.csv) naar dagrendementen, en daarover 
een correlatiematrix berekenen.

**Op te zoeken pandas-operaties:**
- CSV inlezen met de datum als datetime-index
- Van koersen naar dagrendementen (procentuele verandering)
- Correlatie over meerdere kolommen in één keer

In [1]:
import pandas as pd

def load_data(csv_path: str) -> pd.DataFrame:
    """Function to load the dataframe by using the Date column as index and parsing dates to standardise."""
    return pd.read_csv(csv_path, index_col=0, parse_dates=True)

data = load_data('data/prices.csv')

In [2]:
def calculate_return(data: pd.DataFrame) -> pd.DataFrame:
    """Function to return the percentage change per day."""
    return data.pct_change()

returns = calculate_return(data)

    

    

In [3]:
returns.corr()

,EMIM.AS,IUSN.DE,IWDA.AS
EMIM.AS,1.000000,0.732068,0.756767
IUSN.DE,0.732068,1.000000,0.918314
IWDA.AS,0.756767,0.918314,1.000000


In [4]:

returns

,EMIM.AS,IUSN.DE,IWDA.AS
Date,,,
2018-04-26,NaN,NaN,NaN
2018-04-27,0.010412,0.001064,0.004364
2018-04-30,0.005312,0.003899,0.004345
2018-05-02,-0.003894,0.008121,0.001664
2018-05-03,-0.019784,-0.007938,-0.012625
...,...,...,...
2026-06-04,-0.014915,0.005063,-0.000323
2026-06-05,-0.034740,-0.009851,-0.004359
2026-06-08,-0.001826,-0.000791,-0.005514


### Antwoord vraag 1

De correlatiematrix over de dagrendementen (april 2018 t/m juni 2026, 2060 handelsdagen):

|         | EMIM.AS | IUSN.DE | IWDA.AS |
|---------|---------|---------|---------|
| EMIM.AS | 1.00    | 0.73    | 0.76    |
| IUSN.DE | 0.73    | 1.00    | 0.92    |
| IWDA.AS | 0.76    | 0.92    | 1.00    |

**Wat de getallen zeggen.** De correlatie tussen IWDA en IUSN is met 0.92 zeer sterk: 
small caps bewegen op dagbasis vrijwel volledig mee met de wereldindex, ondanks dat er 
geen enkele aandelenoverlap tussen de twee fondsen bestaat. Inhoudelijk verschillende 
fondsen blijken dus niet automatisch gedragsmatig verschillende fondsen. EMIM beweegt 
met 0.73 (t.o.v. IUSN) en 0.76 (t.o.v. IWDA) duidelijk losser: nog steeds sterk 
positief gecorreleerd, maar met merkbaar meer eigen beweging dan small caps.

**Betekenis voor mijn allocatie.** Van mijn twee 20%-posities doet EMIM aantoonbaar 
het meeste diversificatiewerk. De small-cap-positie voegt op dagbasis nauwelijks 
zelfstandige beweging toe; als die positie zijn plek in de portefeuille verdient, 
moet dat uit extra rendement komen en niet uit spreiding. Dat is precies wat vraag 2 
gaat meten.

**Kanttekening bij de maat.** Correlatie meet of fondsen samen bewegen, niet hoe hard. 
IUSN kan voor 92% meebewegen met IWDA en tegelijk volatieler zijn; dat onderscheid 
(samenhang versus beweeglijkheid) komt in vraag 2 aan bod.

**Mijn conclusie:** Het eerste signaal uit het berekenen van de correlatiematrix van deze dataset duidt aan dat ze enigzins tegen verwachting in sterk met elkaar correleren. Voor vraag 2 betekent dit dat ik verwacht dat mijn diversificatie tov een wereldindex de EMIM sterker onafhankelijk beweegt dan de IUSN, en dat dit een iets zwakker effect voor spreiding betekent. Opzich is dit niet super erg want een wereldindex is al een sterke spreiding maar wel een interessante bevinding na het doen van deze analyse. 

## Vraag 2: Mijn mix versus 100% IWDA

**Vraag:** Wat levert mijn 60/20/20-mix (IWDA/EMIM/IUSN) qua rendement en risico op 
vergeleken met simpelweg 100% IWDA over dezelfde periode? Krijg ik betaald voor het 
extra risico, of niet?

**Verwachting vooraf (uit vraag 1):** De spreidingswinst moet vooral van EMIM komen; 
IUSN moet zijn plek eerder in extra rendement bewijzen, want het beweegt vrijwel 
volledig mee met de wereldindex (correlatie 0.92).

**Aanpak:** Van de losse fondsrendementen naar één gewogen portefeuillerendement 
(60% IWDA, 20% EMIM, 20% IUSN). Daarna voor zowel de mix als 100% IWDA twee 
kengetallen berekenen: gemiddeld rendement en standaarddeviatie (risico), beide 
geannualiseerd zodat de getallen iets zeggen op jaarbasis.

**Op te zoeken pandas-operaties:**
- Kolommen wegen en optellen tot één reeks (gewogen som)
- Gemiddelde en standaarddeviatie over een kolom 
- Annualiseren: van dagcijfers naar jaarcijfers 


In [5]:
def weighed_sum(returns: pd.DataFrame, weights: list[float]) -> pd.Series:
    """Bereken het portefeuillerendement per dag als gewogen som van de fondsrendementen; weights volgt de kolomvolgorde van returns."""
    return weights[0] * returns[returns.columns[0]] + weights[1] * returns[returns.columns[1]] + weights[2] * returns[returns.columns[2]] 
    
weighed = weighed_sum(returns, weights=[0.2, 0.2, 0.6])

In [6]:
def geannualiseerde_mean(daily_returns: pd.Series) -> float:
    """Annualiseer het gemiddelde dagrendement naar een verwacht jaarrendement."""
    return daily_returns.mean() * 252 


def geannualiseerde_std(daily_returns: pd.Series) -> float: 
    """Annualiseer de standaarddeviatie van de dagrendementen naar jaarrisico."""
    return daily_returns.std() * (252 ** 0.5)

In [7]:
print(geannualiseerde_mean(weighed), geannualiseerde_std(weighed))
print(geannualiseerde_mean(returns['IWDA.AS']), geannualiseerde_std(returns['IWDA.AS']))

0.12032799453732072 0.15912735581760407
0.1341131899070914 0.1573135467567241


### Antwoord vraag 2

  |              | Rendement/jaar | Risico/jaar |
  |--------------|----------------|-------------|
  | Mix 60/20/20 | 12,0%          | 15,9%       |
  | 100% IWDA    | 13,4%          | 15,7%       |

  Het verschil in rendement is bijna 1,4 procentpunt per jaar, dat vind ik
  veel. In deze periode pakt dat negatief uit voor mijn mix, maar over een
  andere of langere periode kan het ook positief uitvallen. Daarnaast laat de
  standaarddeviatie zien dat de mix een hóger jaarrisico had dan 100%
  wereldindex. Dat risico ben ik bereid te nemen zolang daar mogelijk extra
  rendement tegenover staat. Deze acht jaar hoeven niet te betekenen dat de
  mix altijd minder oplevert.

## Vraag 3: Hoe diep waren de grootste dalingen?

**Vraag:** Hoe diep waren de grootste dalingen van mijn 60/20/20-mix
vergeleken met 100% wereldindex? Niet hoe hard de koers dagelijks trilt
(dat was vraag 2), maar hoeveel procent ik vanaf een piek kwijt was op
het slechtste moment.

**Verwachting vooraf:** Ik denk dat ik verwacht dat het redelijk veel zal zijn, aangezien we een verlies van een aantal procentpunten hebben ten opzichte van de wereldindex. Dus dan moet dit wel een aantal tikken hebben gehad.

**Aanpak:** Van de rendementsreeks naar een cumulatieve waarde-ontwikkeling
(groei van €1), daarna vanaf elke piek de diepste daling bepalen. Hier
verwachtte ik in mijn projectopzet het meeste uitzoekwerk.

**Op te zoeken pandas-operaties:**
- Rendementen cumulatief samenstellen tot een waardereeks (`cumprod`)
- De lopende piek bijhouden (`cummax`)
- De drawdown berekenen en het diepste punt vinden

In [8]:
def drawdown(daily_returns: pd.Series) -> pd.Series:
    """Bereken per dag hoeveel procent de portefeuille onder zijn piek tot dan toe zit (negatief getal)."""
    waarde = (1 + daily_returns).cumprod()
    piek = waarde.cummax()
    return waarde / piek - 1

In [9]:
print(drawdown(weighed).min())
print(drawdown(returns['IWDA.AS']).min())

-0.34584883567820435
-0.3363328555067725



### Antwoord vraag 3

De max drawdown van mijn mix is −34,6% en van 100% IWDA −33,6%. Beide dieptepunten
vallen op 23 maart 2020, dus dat is gewoon de coronacrash. Mijn verwachting dat het
redelijk veel zou zijn klopte dus wel, het zit ruim in de range die ik verwachtte.

Wat ik hier wel opvallend vind: mijn mix zakte zelfs een fractie díeper dan de
wereldindex. Dus de spreiding over drie fondsen heeft me in de crash letterlijk niks
opgeleverd, eerder andersom. Dat sluit eigenlijk precies aan op vraag 1: als alles
met 0.7 a 0.9 met elkaar correleert, dan dalen ze in een crash ook gewoon alle drie
tegelijk. Spreiding binnen aandelen beschermt dus niet tegen een marktbrede klap,
daarvoor zou je echt iets anders moeten toevoegen (obligaties ofzo, maar dat is
buiten scope van dit project).

**Mijn conclusie:** over de hele linie (vraag 2 en 3 samen) heeft mijn 60/20/20-mix
het in deze periode dus slechter gedaan dan simpelweg alles in IWDA: minder rendement,
iets meer risico, en een dieper dal in de crash. Opzich geen reden om gelijk alles om
te gooien want acht jaar is maar één periode, maar ik weet nu wel dat ik −35% moet
kunnen uitzitten zonder panisch te verkopen. Dat vind ik eigenlijk de belangrijkste
les uit deze analyse.

## Vraag 4: Droegen EMIM of IUSN positief bij als IWDA daalde?

**Vraag:** Droegen small caps (IUSN) of opkomende markten (EMIM) positief bij
in periodes waarin IWDA juist daalde? Oftewel: vingen mijn twee 20%-posities
iets op op de slechte dagen van de wereldindex?

**Verwachting vooraf:** Ik verwacht van niet, aangezien de vorige vragen nu wel hebben ertoe geleid dat de diversificatie van mijn portfolio met twee extra aandelen juist heeft geleid tot meer negatieve effecten.

**Aanpak:** Ik selecteer alle dagen met een negatief IWDA-rendement en bekijk
wat EMIM en IUSN op precies die dagen gemiddeld deden.

**Op te zoeken pandas-operaties:**
- Conditioneel selecteren op een DataFrame 
- Gemiddelde per kolom over de gefilterde selectie

In [10]:
def gemiddelde_op_dalingsdagen(returns: pd.DataFrame, benchmark: str) -> pd.Series:
    """Bereken het gemiddelde dagrendement per fonds op de dagen dat de benchmark-kolom een negatief rendement had."""
    dalingsdagen = returns[returns[benchmark] < 0]
    return dalingsdagen.mean()

In [11]:
gemiddelde_op_dalingsdagen(returns, 'IWDA.AS')

EMIM.AS   -0.006581
IUSN.DE   -0.007684
IWDA.AS   -0.007169
dtype: float64

### Antwoord vraag 4

Nee dus. Op de dagen dat IWDA daalde deed EMIM gemiddeld −0,66% en IUSN −0,77%,
tegenover −0,72% voor IWDA zelf. IUSN daalde op die dagen dus gemiddeld zelfs
hárder dan de wereldindex, en EMIM maar een fractie minder hard. Van "opvangen"
is geen sprake, mijn verwachting klopte.

**Mijn conclusie:** dit maakt het beeld van vraag 1 t/m 3 compleet. Mijn twee
20%-posities zijn geen buffer voor slechte dagen maar gewoon extra aandelenrisico
dat meebeweegt met de markt. EMIM dempt nog een héél klein beetje, IUSN versterkt
de klappen eerder. Als ik echt iets wil hebben dat positief bijdraagt als aandelen
dalen, moet dat uit een andere beleggingscategorie komen en niet uit nog meer
aandelen.

## Vraag 5: Voegt mijn 60/20/20-verdeling iets toe?

**Vraag:** Voegt mijn specifieke 60/20/20-verdeling iets toe ten opzichte
van een naïeve 1/3-elk-verdeling, of ten opzichte van 100% wereldindex?

**Verwachting vooraf:** Ik denk van niet, omdat we andere vragen hebben, ja, een soort van hebben aangewezen dat het verwachte effect wat ik van tevoren had niet bereikt, is op een aantal punten. Dus ik denk dat deze analyse daar geen verandering in gaat brengen.

**Aanpak:** Geen nieuwe pandas-operatie. Ik draai mijn bestaande functies
opnieuw met andere gewichten, hier betaalt de herbruikbare opzet uit mijn
projectplan zich terug.

In [12]:
def kengetallen(daily_returns: pd.Series) -> tuple[float, float, float]:
    """Geef rendement/jaar, risico/jaar en max drawdown van een rendementsreeks terug."""
    return geannualiseerde_mean(daily_returns), geannualiseerde_std(daily_returns), drawdown(daily_returns).min()

naief = weighed_sum(returns, weights=[1/3, 1/3, 1/3])

print(kengetallen(naief))
print(kengetallen(weighed))
print(kengetallen(returns['IWDA.AS']))

(np.float64(0.11113786429080695), np.float64(0.16343752095103461), np.float64(-0.35233177285882267))
(np.float64(0.12032799453732072), np.float64(0.15912735581760407), np.float64(-0.34584883567820435))
(np.float64(0.1341131899070914), np.float64(0.1573135467567241), np.float64(-0.3363328555067725))


### Antwoord vraag 5

|                | Rendement/jaar | Risico/jaar | Max drawdown |
|----------------|----------------|-------------|--------------|
| Naïef 1/3 elk  | 11,1%          | 16,3%       | −35,2%       |
| Mix 60/20/20   | 12,0%          | 15,9%       | −34,6%       |
| 100% IWDA      | 13,4%          | 15,7%       | −33,6%       |

Het patroon is verrassend netjes: hoe meer gewicht naar EMIM en IUSN, hoe
slechter alle drie de kengetallen worden. De naïeve 1/3-verdeling verliest
op rendement, risico én drawdown van mijn mix, en mijn mix verliest op
dezelfde drie punten weer van 100% IWDA. Mijn verwachting klopte dus volledig.

**Mijn conclusie:** mijn 60/20/20-verdeling voegt wel iets toe ten opzichte
van naïef gelijk verdelen (zo'n 0,9 procentpunt rendement per jaar extra bij
minder risico), maar dat komt puur doordat ik mínder van de slecht presterende
fondsen vasthoud. De eerlijke conclusie van het hele project is dat in deze
periode elke gram extra EMIM/IUSN waarde kostte. Mijn weging was dus beter dan
naïef wegen, maar het idee erachter (extra rendement of spreiding halen uit
small caps en opkomende markten) heeft zich in deze acht jaar niet bewezen.